# 00 - Calibrate DINO

Step 00 of the TamIA pipeline in Colab format. Runs `calibrate_dino.py` on the validation split to inspect DINO proposal recall versus false-positive boxes.

Run this notebook with a GPU runtime. The code executes the same root script used by the TamIA Slurm job.

In [1]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

# Edit these if needed.
REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "main"
PROJECT_DIR = Path("/content/INF8225_Projet")

# If your Drive folder is not auto-detected by experiments/colab_setup.py, set it explicitly,
# for example: "/content/drive/MyDrive/Projet_Medsam".
DRIVE_FOLDER = None

# Keep INSTALL_DEPS=True on the first notebook in a fresh Colab runtime.
# You can set it to False for later notebooks in the same runtime.
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from experiments.colab_setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())


→ detect GPU
✓ deps already installed, skipping
✓  /content/INF8225_Projet/data already linked
✓  /content/INF8225_Projet/work_dirs already linked
✓  /content/INF8225_Projet/MedSAM/work_dir already linked
✓  /content/INF8225_Projet/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth already linked

Project : /content/INF8225_Projet
Drive   : /content/drive/MyDrive/Projet_Medsam
Device  : Tesla T4
Torch   : 2.4.0+cu121

⚠  numpy was already loaded in this kernel before setup reinstalled it.
   Runtime → Restart session, then run your imports again
   (no need to rerun setup — deps are pinned on disk).
cwd: /content/INF8225_Projet


In [2]:
#@title Verify shared assets
from pathlib import Path

required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"


OK      data/MSD_pancreas/train.json
OK      data/MSD_pancreas/val.json
OK      data/MSD_pancreas/test.json
OK      work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth


In [3]:
#@title Run pipeline step
from collections import deque
import subprocess
import sys

SCRIPT = "experiments.msd.resnet18_recall.calibrate_detector"
cmd = [sys.executable, "-u", "-m", SCRIPT]
print("Running:", " ".join(cmd))

last_lines = deque(maxlen=160)
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    last_lines.append(line)
return_code = proc.wait()

if return_code != 0:
    print("\n" + "=" * 80)
    print(f"{SCRIPT} failed with exit code {return_code}. Last captured lines:")
    print("=" * 80)
    print("".join(last_lines))
    raise RuntimeError(f"{SCRIPT} failed with exit code {return_code}")


Running: /usr/bin/python3 -u calibrate_dino.py
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
Appareil detecte : cuda
Loads checkpoint by local backend from path: work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth
/usr/local/lib/python3.12/dist-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This lim

In [4]:
#@title Expected output
print("No artifact is required for step 00. Check the printed TP/FN/FP table above.")


No artifact is required for step 00. Check the printed TP/FN/FP table above.
